In [ ]:
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producent = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

try:
    with open('klienci_banku.json', 'r', encoding='utf-8') as f:
        lista_klientow = json.load(f)
    print(f"Pomyślnie wczytano {len(lista_klientow)} klientów z pliku.")
except FileNotFoundError:
    print("Błąd: Nie znaleziono pliku klienci_banku.json! Upewnij się, że plik jest w tym samym folderze.")
    lista_klientow = []

rodzaje_transakcji = [
    "transakcja krajowa",
    "transakcja zagraniczna",
    "wypłata z bankomatu",
    "płatność kartą",
    "przelew internetowy",
    "przelew ekspresowy",
    "płatność mobilna",
    "zakup online",
    "płatność zbliżeniowa",
    "przelew na telefon"]

miasta = [
    'Warszawa', 'Kraków', 'Łódź', 'Wrocław', 'Poznań', 
    'Gdańsk', 'Szczecin', 'Bydgoszcz', 'Lublin', 'Białystok', 
    'Katowice', 'Gdynia', 'Częstochowa', 'Radom', 'Sosnowiec', 
    'Toruń', 'Kielce', 'Rzeszów', 'Gliwice', 'Zabrze', 
    'Olsztyn', 'Bielsko-Biała', 'Bytom', 'Zielona Góra', 'Rybnik', 
    'Ruda Śląska', 'Tychy', 'Gorzów Wielkopolski', 'Dąbrowa Górnicza', 'Elbląg',  
    'Siedlce', 'Mysłowice', 'Konin', 'Piła', 'Piotrków Trybunalski','Londyn', 'Paryż', 'Berlin', 'Madryt', 'Rzym', 'Praga', 'Wiedeń', 'Amsterdam', 'Lizbona', 'Ateny',
    'Oslo', 'Sztokholm', 'Helsinki', 'Kopenhaga', 'Bruksela', 'Dublin', 'Budapeszt', 'Bratysława', 'Tallinn', 'Ryga',
    'Wilno', 'Zurych', 'Genewa', 'Monako', 'Reykjavik', 'Nowy Jork', 'Los Angeles', 'Chicago', 'Toronto', 'Vancouver',
    'Meksyk', 'Hawana', 'Waszyngton', 'Miami', 'San Francisco', 'Montreal', 'Rio de Janeiro', 'São Paulo', 'Buenos Aires', 'Santiago',
    'Lima', 'Bogota', 'Quito', 'Caracas', 'Montevideo', 'Tokio', 'Seul', 'Pekin', 'Szanghaj', 'Bangkok',
    'Singapur', 'Dubaj', 'Delhi', 'Bombaj', 'Hanoi', 'Dżakarta', 'Kuala Lumpur', 'Tajpej', 'Tel Awiw', 'Manila',
    'Rijad', 'Kair', 'Kapsztad', 'Nairobi', 'Casablanca', 'Lagos', 'Addis Abeba', 'Tunis', 'Marrakesz', 'Johannesburg',
    'Dakar', 'Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Auckland', 'Wellington', 'Luksemburg', 'Andora', 'Belgrad',
    'Zagrzeb', 'Sarajewo', 'Skopje', 'Tirana', 'Erewań', 'Tbilisi', 'Baku', 'Astana', 'Taszkent', 'Maskat',]


def generate_bank_transaction():
    klient=random.choice(lista_klientow)

    fraud = random.randint(1, 50) == 1
    if fraud:
        return {
        'tx_id': f'BANK-{random.getrandbits(32)}', 
        'user_id': klient.get('id'),
        'amount': round(random.uniform(klient.get('saldo')-klient.get('saldo')/2, klient.get('saldo')), 2), 
        'timestamp': datetime.now().isoformat(),
        'location': random.choice(miasta),
        'tx_type': random.choice(rodzaje_transakcji),
        'currency': random.choice(['PLN','EUR','USD','JPY','INR']),
        'fraud':1,
        'balance':klient.get('saldo')
        }
    else:
        return {
        'tx_id': f'BANK-{random.getrandbits(32)}', 
        'user_id': klient.get('id'), 
        'amount': round(random.uniform(1.0, klient.get('saldo')), 2),
        'timestamp': datetime.now().isoformat(),
        'location': klient.get('miasto'),
        'tx_type': random.choice(rodzaje_transakcji),
        'currency': 'PLN',
        'fraud':0,
        'balance':klient.get('saldo')
    }

print("Rozpoczynanie nadawania transakcji bankowych...")

i = 0 
try:
    while True:
        tx = generate_bank_transaction()
        
        producent.send('bank-transactions', value=tx)
        if tx['fraud']==1:
            print(f"[{i+1:03d}] ID: {tx['tx_id']}")
            print(f"      Użytkownik: {tx['user_id']}| Saldo: {tx['balance']}")
            print(f"      Lokalizacja: {tx['location']} | Typ: {tx['tx_type']}")
            print(f"      Kwota(PLN): {tx['amount']:>8.2f} waluta transakcji: {tx['currency']}")
        
        time.sleep(1) 
        i += 1

except KeyboardInterrupt:
    print("\nZatrzymano przez użytkownika.")

finally:
    producent.flush()
    producent.close()
    print("Połączenie z Kafką zamknięte.")

Pomyślnie wczytano 100 klientów z pliku.
Rozpoczynanie nadawania transakcji bankowych...
[104] ID: BANK-227986075
      Użytkownik: 9748d4eb| Saldo: 9857.1
      Lokalizacja: Szanghaj | Typ: płatność kartą
      Kwota(PLN):  5622.13 waluta transakcji: USD
[108] ID: BANK-3128477434
      Użytkownik: df3fb918| Saldo: 132053.76
      Lokalizacja: Kuala Lumpur | Typ: przelew na telefon
      Kwota(PLN): 100261.28 waluta transakcji: INR
[243] ID: BANK-4207681234
      Użytkownik: a907a0c6| Saldo: 58381.06
      Lokalizacja: Auckland | Typ: transakcja krajowa
      Kwota(PLN): 47472.40 waluta transakcji: INR
[290] ID: BANK-682488614
      Użytkownik: 1876d0a1| Saldo: 143276.86
      Lokalizacja: Manila | Typ: transakcja zagraniczna
      Kwota(PLN): 86964.14 waluta transakcji: JPY
[301] ID: BANK-2900133314
      Użytkownik: 8124a172| Saldo: 30677.76
      Lokalizacja: Nowy Jork | Typ: przelew na telefon
      Kwota(PLN): 22194.22 waluta transakcji: INR
[367] ID: BANK-3175997450
      Użytkow